In [12]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [13]:
spark = SparkSession.builder.appName("GTFS Silver").getOrCreate()

In [14]:
runid = "2026-05-13_12-43-46"

In [15]:
# Read the Parquet files from Silver layer
df_stop = spark.read.parquet(f"../../../data/silver/gtfs_static/stops")
df_trip = spark.read.parquet(f"../../../data/silver/gtfs_static/trips")
df_stop_times = spark.read.parquet(f"../../../data/silver/gtfs_static/stop_times")
df_route = spark.read.parquet(f"../../../data/silver/gtfs_static/routes")
df_calendar = spark.read.parquet(f"../../../data/silver/gtfs_static/calendar_dates")

In [16]:
df_total = (
    df_stop_times.join(df_trip, "trip_id")
    .join(df_route, "route_id")
    .join(df_calendar, "service_id")
    .join(df_stop, "stop_id")
)

In [17]:
df_total = (
    df_total
    .withColumn("date", F.col("date").cast("string"))
    .withColumn("date", F.to_date("date", "yyyyMMdd"))
)

In [ ]:
  
df_test = (
    df_total
    .filter(
        (F.col("date") == "2026-05-19") 
    )
)

In [19]:
df_total.write.mode("overwrite").parquet(f"../../../data/gold/gtfs_static/total")
df_test.write.mode("overwrite").parquet(f"../../../data/gold/gtfs_static/total_test")

26/05/13 16:57:50 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/05/13 16:57:50 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/05/13 16:57:50 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/05/13 16:57:50 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/05/13 16:57:50 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/05/13 16:57:50 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 58.46% for 13 writers
26/05/13 16:57:50 WARN MemoryManager: Total allocation exceeds 95.

In [20]:
df_test.show(5, truncate=False)

+---------------------------+----------+----------------------------------------------+-----------------------------------------------------------------------------------------------------+------------+--------------+-------------+----------------+---------------+----------+--------------+----------------------+---------+---------+
|stop_id                    |service_id|route_id                                      |trip_id                                                                                              |arrival_time|departure_time|stop_sequence|route_short_name|route_long_name|date      |exception_type|stop_name             |stop_lat |stop_lon |
+---------------------------+----------+----------------------------------------------+-----------------------------------------------------------------------------------------------------+------------+--------------+-------------+----------------+---------------+----------+--------------+----------------------+---------+---------